# Week 01 — Concepts and Geographic Questions

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Introduction to Data Science · Concept, geometry, attributes, CRS, provenance

**Learning outcome.** Interpret maps and check unique cell-date observations.

**What you will implement:** Dataframe keys, vector/raster concepts, simple thematic mapping.  
**Mathematical model:** Cell area A = width × height; coordinate reference systems.

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** Cell area A = width × height; coordinate reference systems

**Method family:** Dataframe keys, vector/raster concepts, simple thematic mapping.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
cells = pd.read_csv(DATA/'cells.csv')
raw = pd.read_csv(DATA/'observations_raw.csv')
obs = raw.drop_duplicates()
print('Grid:', len(cells), 'Raw:', len(raw), 'Unique:', len(obs))
assert len(cells)==144 and len(obs)==864
assert not obs.duplicated(['cell_id','date']).any()
print(cells[['cell_id','easting_m','northing_m','sector']].head())

### Reading the main code

**Read and audit.** `pd.read_csv` loads tables; `drop_duplicates()` removes identical complete rows; `assert` checks both expected counts and uniqueness of `(cell_id, date)`.

- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `assert` — Checks a claimed condition and stops if it is false.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
june = cells.merge(obs[obs.date=='2025-06-01'],on='cell_id',validate='one_to_one')
arr = june.sort_values(['row','col']).lst_c.to_numpy().reshape(12,12)
plt.figure(figsize=(7,5)); plt.imshow(arr,origin='lower',cmap='RdYlBu_r')
plt.colorbar(label='Synthetic land surface temperature (°C)')
plt.title('June 2025 · Synthetic Teaching Grid');plt.xlabel('Column');plt.ylabel('Row');plt.show()
print('Mean °C:', round(june.lst_c.mean(),2))

### Reading the output code

**Make the map.** Merge one June record per cell, sort by grid row/column, reshape to a 12×12 array, then display LST in degrees Celsius with the south-up origin.

- `merge` — Joins tables on a declared key; validate cardinality to avoid duplicate matches.
- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Interpretation.** The map is one synthetic observation period; original information support remains 250 m per cell even if a raster display is enlarged.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Explain exactly what one observation represents, and what information is lost if geometry is not joined correctly.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.